# Infaction — City Placement LOCKED v0.4.6

Runs existing Faction/Zone/Location generation and Location→Zone placement unchanged; then applies **City Placement LOCKED** as an overlay. All selected Zones belong to one city; standalone Location can be Inside or Outside separately. Site generation is intentionally **not yet implemented**.

Sources: [City engine](https://github.com/playrenathea/rantau/blob/main/infaction_city_locked.py), [original City scores](https://github.com/playrenathea/rantau/blob/main/data/infaction_city_score_original_locked.json), [placement engine](https://github.com/playrenathea/rantau/blob/main/infaction_placement_locked.py), [master Drive specification](https://docs.google.com/document/d/15AVs2_mh66o-M8YnetGeLZqB2HuXRm04ZqIhNWWHT6s/edit).

In [ ]:
import urllib.request, json, contextlib, io, types
ROOT='https://raw.githubusercontent.com/playrenathea/rantau/main/'
def get_text(path): return urllib.request.urlopen(ROOT+path,timeout=30).read().decode('utf-8')
source=json.loads(get_text('notebooks/Infaction_Faction_Naming_Zone_Location_Generator_v0.4.3_City_Experiment_Revised.ipynb'))
ns={'__name__':'__main__'}
for i in (1,2,3,5,8,11):
    with contextlib.redirect_stdout(io.StringIO()):
        exec(compile(''.join(source['cells'][i]['source']),str(i),'exec'),ns)
placement=types.ModuleType('infaction_placement_locked')
exec(compile(get_text('infaction_placement_locked.py'),'infaction_placement_locked.py','exec'),placement.__dict__)
city=types.ModuleType('infaction_city_locked')
exec(compile(get_text('infaction_city_locked.py'),'infaction_city_locked.py','exec'),city.__dict__)
neutral_data=json.loads(get_text('data/infaction_neutral_static_v2_locked.json'))
neutral={r['place']:r['neutral_affinity'] for r in neutral_data['records']}
city_scores=json.loads(get_text('data/infaction_city_score_original_locked.json'))
assert len(neutral)==len(city_scores)==110
print('Ready:',placement.VERSION,city.VERSION)

In [ ]:
SEED=8100000
world=ns['generate_world_with_city_zones'](SEED)
placed=placement.place_world(world,ns['ZONE_DATA'],ns['LOCATION_DATA'],neutral,seed=SEED)
result=city.city_place_world(placed,city_scores,seed=SEED)
print('Inside Zones:',result['inside_zones'])
print('Outside Zones:',result['outside_zones'])
print('Outside extra:',result['extra_outside_zone'])
print('Standalone Locations:',result['standalone_locations'])
print('All Locations city-tagged:',len(result['location_city']))
assert len(result['location_city'])==len(placed['locations'])

In [ ]:
from collections import Counter
N=100
stats=Counter()
for seed in range(8100000,8100000+N):
    w=ns['generate_world_with_city_zones'](seed)
    p=placement.place_world(w,ns['ZONE_DATA'],ns['LOCATION_DATA'],neutral,seed=seed)
    c=city.city_place_world(p,city_scores,seed=seed)
    stats.update({'worlds':1, 'outside_zones':c['outside_zone_count'],
        'standalone':len(c['standalone_locations']),
        'standalone_outside':sum(v['city']=='Outside' for v in c['standalone_locations'].values())})
    assert c['outside_zone_count']>=1
    assert set(c['standalone_locations'])==set(p['outside'])
    assert all(c['location_city'][loc]==c['zones'][v['zone']]['city'] for loc,v in p['placement'].items() if v['zone'] is not None)
    assert all(c['zones'][z]['city']=='Outside' for z in p['zones'] if c['zones'][z]['base_city_score']==0)
print('100-world smoke test:',dict(stats))

In [ ]:
from pathlib import Path
Path(f'infaction_world_city_{SEED}.json').write_text(json.dumps({'placement':placed,'city':result},ensure_ascii=False,indent=2))
print('City overlay exported; site grouping pending')